In [50]:
import pandas as pd
import timeit

-  Здійснити data cleaning
-  Окремими функціями сформувати вибірки:
   - Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.
   - Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.
   - Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії
   - Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [51]:
df=pd.read_csv("household_power_consumption.txt", sep=";", low_memory=False)
df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)

def more_than_5kw(df):
    return df[df["Global_active_power"].astype(float) > 5.0]

def current_between_19_and_20(df):
    df_current = df[(df["Global_intensity"].astype(float).between(19, 20))]
    df_current = df_current[(df_current["Sub_metering_2"].astype(float) > (df_current["Sub_metering_1"].astype(float) + df_current["Sub_metering_3"].astype(float)))]  

def random_sample(df):
    df_sample = df.sample(n=500000, random_state=42, replace=False)  

    mean_sub_metering_1 = df_sample["Sub_metering_1"].astype(float).mean()
    mean_sub_metering_2 = df_sample["Sub_metering_2"].astype(float).mean()
    mean_sub_metering_3 = df_sample["Sub_metering_3"].astype(float).mean()

    print("Mean Sub_metering_1:", mean_sub_metering_1)
    print("Mean Sub_metering_2:", mean_sub_metering_2)
    print("Mean Sub_metering_3:", mean_sub_metering_3)
    return df_sample

def after_1800(df):
    df_after_1800 = df[df["Time"].str.startswith(("18:", "19:", "20:", "21:", "22:", "23:"))]
    df_after_1800 = df_after_1800[df_after_1800["Global_active_power"].astype(float) > 6.0]
    
    group_1 = df_after_1800["Sub_metering_1"].astype(float)
    group_2 = df_after_1800["Sub_metering_2"].astype(float)
    group_3 = df_after_1800["Sub_metering_3"].astype(float)

    df_evening_group2 = df_after_1800[(group_2 > group_1) & (group_2 > group_3)]

    half = len(df_evening_group2) // 2
    first_half = df_evening_group2.iloc[:half]
    second_half = df_evening_group2.iloc[half:]

    result = pd.concat([first_half.iloc[::3],second_half.iloc[::4]])

    return result

- Пронормувати та стандартизувати вибраний датасет
- Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.
- Провести One Hot Encoding категоріального атрибута.

In [52]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", 
                "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

df[numeric_cols] = df[numeric_cols].astype(float)

minmax_scaler = MinMaxScaler()
df_normalized = pd.DataFrame(minmax_scaler.fit_transform(df[numeric_cols]), columns=[col + "_norm" for col in numeric_cols])

standard_scaler = StandardScaler()
df_standardized = pd.DataFrame(standard_scaler.fit_transform(df[numeric_cols]), columns=[col + "_std" for col in numeric_cols])

pearson_corr = df["Global_active_power"].corr(df["Voltage"], method="pearson")
spearman_corr = df["Global_active_power"].corr(df["Voltage"], method="spearman")
print("Pearson correlation (Global_active_power & Voltage):", pearson_corr)
print("Spearman correlation (Global_active_power & Voltage):", spearman_corr)

df_encoded = pd.get_dummies(df, columns=["Date", "Time"], drop_first=True)

print(df_encoded)


Pearson correlation (Global_active_power & Voltage): -0.3997616096289585
Spearman correlation (Global_active_power & Voltage): -0.32521294259808614
         Global_active_power  Global_reactive_power  Voltage  \
0                      4.216                  0.418   234.84   
1                      5.360                  0.436   233.63   
2                      5.374                  0.498   233.29   
3                      5.388                  0.502   233.74   
4                      3.666                  0.528   235.68   
...                      ...                    ...      ...   
2075254                0.946                  0.000   240.43   
2075255                0.944                  0.000   240.00   
2075256                0.938                  0.000   239.82   
2075257                0.934                  0.000   239.70   
2075258                0.932                  0.000   239.55   

         Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
0     